In [55]:
from PIL import Image, ImageDraw 

In [56]:
def UpscalingImage2x(image):
    width = image.size[0] #Определяем ширину. 
    height = image.size[1] #Определяем высоту. 
    
    up_image = Image.new(mode="RGB", size=(width * 2, height * 2))
    draw = ImageDraw.Draw(up_image) #Создаем инструмент для рисования. 
    	
    orig_pix = image.load() #Выгружаем значения пикселей.
    
    for i in range(0, width):
        for j in range(0, height):
            pixel = orig_pix[i, j]
            
            for k in range(0, 2):
                for m in range(0, 2):
                    draw.point((2 * i + k, 2 * j + m), pixel)
                    
    return up_image

In [57]:
image = Image.open("./data/test_img.jpg") #Открываем изображение. 
image.size

(736, 981)

In [94]:
up_image = UpscalingImage2x(image)
up_image.show()

up_image.save('./data/image_U2x.jpg')
up_image.size

(1472, 1962)

In [59]:
def UpscalingImage2xSmoothing(image):
    width = image.size[0] #Определяем ширину. 
    height = image.size[1] #Определяем высоту. 
    
    up_image = Image.new(mode="RGB", size=(height * 2, width * 2))
    draw = ImageDraw.Draw(up_image) #Создаем инструмент для рисования. 
    	
    orig_pix = image.load() #Выгружаем значения пикселей.
    
    for i in range(0, height - 1):
        for j in range(0, width - 1):
            pixel = orig_pix[i, j]
            
            draw.point((2 * i, 2 * j), pixel)
                    
    

In [ ]:
width = image.size[0] #Определяем ширину. 
height = image.size[1] #Определяем высоту. 

up_image = Image.new(mode="RGB", size=(width * 2, height * 2), color=(255, 255, 255))
draw = ImageDraw.Draw(up_image) #Создаем инструмент для рисования. 
	
orig_pix = image.load() #Выгружаем значения пикселей.

# Заполнение до последнего столбца
for i in range(0, height - 1):
    for j in range(0, width - 1):
        pixel = orig_pix[j,i]
        draw.point((2 * j, 2 * i), pixel)

# заполнение последних строк и столбцов
for i in range(height):
    pixel = orig_pix[width - 1, i]
    draw.point((2 * width - 1, 2 * i), pixel)
    
for i in range(width):
    pixel = orig_pix[i, height - 1]
    draw.point((2 * i, 2 * height - 1), pixel)
    
draw.point((2 * width - 1, 2 * height - 1), orig_pix[width - 1, height - 1])

# создадим мнимый столбец
im_comlumn = []
for j in range(height):
    av_pix = [255, 255, 255]
    pixel_l = orig_pix[width - 2, j]
    pixel_r = orig_pix[width - 1, j]
    av_pix = [(pixel_l[k] + pixel_r[k]) // 2 for k in range(3)]
    
    im_comlumn.append(av_pix)
    #im_comlumn.append([255, 255, 255])

# создадим мнимую строку
im_row = []
for i in range(width):
    im_pix = [255, 255, 255]
    
    pixel_t = orig_pix[i, height - 2]
    pixel_d = orig_pix[i, height - 1]
    im_pix = [(pixel_t[k] + pixel_d[k]) // 2 for k in range(3)]
    
    im_row.append(im_pix)
        
    #im_row.append([255, 255, 255])
    

pixels = up_image.load()

def is_index_hor(i, j):
    if (i % 2 == 1 or i >= 2 * width - 3) and (j % 2 == 0 or j == 2 * height - 1):
        return True
    return False
def is_index_vert(i, j):
    if (j % 2 == 1 or j >= 2 * height - 3) and (i % 2 == 0 or i == 2 * width - 1):
        return True
    return False

# Заполнение средних пикселей по горизонтали
for i in range(2 * width - 1):
    for j in range(2 * height):
        
        # Заполняем пиксель по горизонтали
        if is_index_hor(i, j):
            pixel_l = pixels[i - 1, j]
            pixel_r = pixels[i + 1, j]
            
            if i == 2 * width - 3:
                pixel_r = im_comlumn[j // 2]
            if i == 2 * width - 2:
                pixel_l = im_comlumn[j // 2]
                
            av_pix = [(pixel_l[k] + pixel_r[k]) // 2 for k in range(3)]
            draw.point((i, j), (av_pix[0], av_pix[1], av_pix[2]))
            
# Заполнение средних пикселей по вертикали
for i in range(2 * width):
    for j in range(2 * height - 1):
        
        if is_index_vert(i, j):
            pixel_u = pixels[i, j - 1]
            pixel_d = pixels[i, j + 1]
            
            if j == 2 * height - 3:
                pixel_d = im_row[i // 2]
            if j == 2 * height - 2:
                pixel_t = im_row[i // 2]
            
            av_pix = [(pixel_u[k] + pixel_d[k]) // 2 for k in range(3)]
            draw.point((i, j), (av_pix[0], av_pix[1], av_pix[2]))
        

        
        
# up_image.show()

def is_central_pix(i, j):
    if (i % 2 == 1 and j % 2 == 1) or ((i == 2 * width - 3 or i == 2 * width - 2) and j % 2 == 1) or (i % 2 == 1 and (j == 2 * height - 3 or j == 2 * height - 2)):
        return True
    return False

# Заполняем центральные клетки     
for i in range(1, 2 * width - 1):   
    for j in range(1, 2 * height - 1):
        
        if is_central_pix(i, j):
            pixel_lt = pixels[i - 1, j - 1] # left top
            pixel_ld = pixels[i - 1, j + 1] # left down
            pixel_rt = pixels[i + 1, j - 1] 
            pixel_rd = pixels[i + 1, j + 1]
            # print(i, j)
            
            if i == 2 * width - 3:
                pixel_rt = im_comlumn[(j - 1) // 2]
                pixel_rd = im_comlumn[(j - 1) // 2 + 1]
                
            if i == 2 * width - 2:
                pixel_lt = im_comlumn[(j - 1) // 2]
                pixel_ld = im_comlumn[(j - 1) // 2 + 1]
                
            if j == 2 * height - 3:
                pixel_ld = im_row[(i - 1) // 2]
                pixel_rd = im_row[(i - 1) // 2 + 1]
                
            if j == 2 * height - 2:
                pixel_lt = im_row[(i - 1) // 2]
                pixel_rt = im_row[(i - 1) // 2 + 1]
            
            av_pix = [
                (
                    pixel_lt[k] + 
                    pixel_ld[k] + 
                    pixel_rt[k] + 
                    pixel_rd[k]
                ) // 4 for k in range(3)]
            
            draw.point((i, j), (av_pix[0], av_pix[1], av_pix[2]))
    
up_image.show()
up_image.save('./data/image_U2xS.png')

In [67]:
def get_index_in_im(ind):
    return ind // 2

In [136]:
get_index_in_im(8)

4

In [118]:
is_central_pix(1469, 1)

True

In [143]:
is_index_vert(1468, 1960)

True

In [146]:
is_index_hor(1469, 2)

True